# BCEDF: Breast Cancer Early Detection Framework
## Full Training Pipeline for M.S. Thesis

**Runtime setup**: Go to Runtime → Change runtime type → T4 GPU

**Before running**: Upload these folders to MyDrive/Project/:
- `breast_cancer_project/` (this project)
- `breakhis/` (BreakHis dataset)
- `mias/` (MIAS dataset)
- `inbreast/` (INbreast dataset)

In [ ]:
# Cell 1: Mount Drive
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted')

In [ ]:
# Cell 2: Copy project from Drive to Colab local
import shutil, sys, os

src = '/content/drive/MyDrive/Project/breast_cancer_project'
dst = '/content/Project/breast_cancer_project'
if os.path.exists(dst):
    shutil.rmtree(dst)
shutil.copytree(src, dst)
print(f'Project copied to {dst}')
print(f'src/ exists: {os.path.exists(os.path.join(dst, "src"))}')
sys.path.insert(0, dst)

In [ ]:
# Cell 3: Install dependencies
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118
!pip install -q numpy pandas scikit-learn scipy matplotlib seaborn opencv-python albumentations tqdm tensorboard PyYAML easydict pydicom --no-build-isolation
print('Dependencies installed')

In [ ]:
# Cell 4: Verify GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

In [ ]:
# Cell 5: Configuration - EDIT YOUR PATHS HERE
PROJECT_DIR = '/content/drive/MyDrive/Project/breast_cancer_project'

DATASETS = {
    'breakhis': '/content/drive/MyDrive/Project/breakhis/BreaKHis_v1/BreaKHis_v1/histology_slides/breast',
    'mias': '/content/drive/MyDrive/Project/mias',
    'inbreast': '/content/drive/MyDrive/Project/inbreast'
}

ARCHITECTURES = ['cnn', 'resnet50', 'densenet121']
DATASET_CONFIGS = [['breakhis'], ['mias'], ['inbreast'], ['breakhis', 'mias', 'inbreast']]
EPOCHS, BATCH_SIZE, LEARNING_RATE = 50, 32, 0.001

In [ ]:
# Cell 6: Imports
import sys, os, json, yaml, torch, numpy as np
sys.path.insert(0, '/content/Project/breast_cancer_project')
from datetime import datetime
from easydict import EasyDict as edict
from src.models.model_factory import create_model, ModelFactory, CustomCNN
from src.datasets.dataloader import DataLoaderFactory
from src.training.trainer import Trainer
from src.evaluation.metrics import ClassificationMetrics
from src.evaluation.visualizer import Visualizer
from src.evaluation.gradcam import GradCAMView
print('All imports successful')

In [ ]:
# Cell 7: Helper functions
def create_config_for_run(model_name, dataset_names, epochs=50, batch_size=32, lr=0.001):
    cfg = {
        'data': {
            'breakhis': {'path': DATASETS['breakhis'], 'use': 'breakhis' in dataset_names, 'magnification': [40, 100, 200, 400]},
            'inbreast': {'path': DATASETS['inbreast'], 'use': 'inbreast' in dataset_names},
            'mias': {'path': DATASETS['mias'], 'use': 'mias' in dataset_names}
        },
        'training': {'batch_size': batch_size, 'epochs': epochs, 'early_stop_patience': 10, 'learning_rate': lr,
            'min_lr': 1e-6, 'weight_decay': 0.0001, 'num_workers': 2, 'mixed_precision': True,
            'label_smoothing': 0.1, 'k_folds': 0, 'val_split': 0.15, 'test_split': 0.10, 'seed': 42, 'gradient_clip': 1.0},
        'model': {'name': model_name, 'pretrained': model_name != 'cnn', 'num_classes': 2, 'dropout': 0.3, 'use_ensemble': False},
        'augmentation': {'img_size': 224, 'clahe_clip_limit': 2.0, 'clahe_grid_size': 8,
            'mixup_alpha': 0.2, 'cutmix_alpha': 1.0, 'mixup_prob': 0.5,
            'rotation': 30, 'brightness': 0.2, 'contrast': 0.2, 'saturation': 0.2, 'hue': 0.1},
        'device': torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
        'output': {'checkpoint_dir': f'/content/outputs/{model_name}/checkpoints', 'log_dir': f'/content/outputs/{model_name}/logs',
            'plot_dir': f'/content/outputs/{model_name}/plots', 'model_dir': f'/content/outputs/{model_name}/models',
            'gradcam_dir': f'/content/outputs/{model_name}/gradcam'}
    }
    cfg_obj = edict(cfg)
    cfg_obj.cfg = cfg_obj
    return cfg_obj

def set_seed(seed):
    import random
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    np.random.seed(seed); random.seed(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

print('Helper functions defined')

In [ ]:
# Cell 8: Train function
def train_and_evaluate(model_name, dataset_names, results_dict):
    ds_label = '+'.join(dataset_names)
    run_name = f'{model_name}_{ds_label}'
    print(f'\n{"="*60}\nRUN: {run_name}\n{"="*60}')
    cfg = create_config_for_run(model_name, dataset_names, epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LEARNING_RATE)
    set_seed(cfg.training.seed)
    for d in ['checkpoint_dir', 'log_dir', 'plot_dir', 'model_dir', 'gradcam_dir']:
        os.makedirs(cfg.output[d], exist_ok=True)
    model = create_model(cfg).to(cfg.device)
    print(f'Params: {sum(p.numel() for p in model.parameters()):,}')
    loader_factory = DataLoaderFactory(cfg)
    train_loader, val_loader, test_loader = loader_factory.get_dataloaders(dataset_names)
    print(f'Train: {len(train_loader.dataset)} | Val: {len(val_loader.dataset)} | Test: {len(test_loader.dataset)}')
    trainer = Trainer(model, cfg)
    best_val_acc = trainer.fit(train_loader, val_loader)
    print(f'Best val acc: {best_val_acc:.4f}')
    test_loss, test_acc, test_info = trainer.validate(test_loader)
    metrics = ClassificationMetrics(num_classes=2)
    all_outputs, all_labels = [], []
    model.eval()
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(cfg.device), labels.to(cfg.device)
            outputs = torch.softmax(model(images), dim=1)
            all_outputs.append(outputs.cpu().numpy()); all_labels.append(labels.cpu().numpy())
    all_outputs, all_labels = np.concatenate(all_outputs), np.concatenate(all_labels)
    report = metrics.compute(all_labels, all_outputs.argmax(axis=1), all_outputs)
    print(f'Test Acc: {report["accuracy"]:.4f} | F1: {report["f1_score"]:.4f} | AUC: {report["auc_roc"]:.4f}')
    viz = Visualizer(cfg.output.plot_dir)
    try:
        viz.plot_confusion_matrix(np.array(report['confusion_matrix']), class_names=['benign', 'malignant'])
        viz.plot_roc_curve(all_labels, all_outputs, class_names=['benign', 'malignant'])
        viz.plot_training_history(trainer.train_losses, trainer.val_losses, trainer.train_accs, trainer.val_accs)
    except Exception as e: print(f'Plot warning: {e}')
    try:
        target_layers = {'resnet50': ['layer4'], 'densenet121': ['features'], 'cnn': ['features']}
        gradcam_view = GradCAMView(model, target_layers.get(model_name, ['features']), cfg.device)
        import cv2
        for idx in range(min(10, len(test_loader.dataset))):
            img_tensor = test_loader.dataset[idx][0].unsqueeze(0).to(cfg.device)
            heatmaps = gradcam_view.generate_heatmap(img_tensor)
            img_np = (img_tensor.squeeze().cpu().numpy().transpose(1,2,0) * 255).astype(np.uint8)
            for layer_name, heatmap in heatmaps.items():
                gradcam_view.save_heatmap(img_np, heatmap, os.path.join(cfg.output.gradcam_dir, f'sample_{idx}_{layer_name}.png'))
        print(f'Grad-CAM: 10 samples')
    except Exception as e: print(f'Grad-CAM warning: {e}')
    torch.save({'model_state_dict': model.state_dict(), 'test_accuracy': report['accuracy']},
        os.path.join(cfg.output.model_dir, f'{model_name}_final.pth'))
    results_dict[run_name] = {'model': model_name, 'datasets': ds_label, 'accuracy': float(report['accuracy']),
        'f1_score': float(report['f1_score']), 'auc_roc': float(report['auc_roc']),
        'sensitivity': float(report['sensitivity']), 'specificity': float(report['specificity']),
        'best_val_acc': float(best_val_acc)}
    return results_dict

print('Train function defined')

In [ ]:
# Cell 9: Run the full pipeline
results = {}
start_time = datetime.now()
print(f'Start: {start_time}')
print(f'Architectures: {ARCHITECTURES}')
print(f'Dataset configs: {DATASET_CONFIGS}')
print(f'Epochs: {EPOCHS}, Batch: {BATCH_SIZE}, LR: {LEARNING_RATE}')

for model_name in ARCHITECTURES:
    for ds_names in DATASET_CONFIGS:
        try:
            if ds_names[0] in DATASETS:
                p = DATASETS[ds_names[0]]
                print(f'Checking {ds_names[0]} path: {p} -> exists: {os.path.exists(p)}')
            results = train_and_evaluate(model_name, ds_names, results)
        except Exception as e:
            import traceback; traceback.print_exc()
            print(f'FAILED: {model_name} on {ds_names}: {e}')

elapsed = datetime.now() - start_time
print(f'\n{"="*60}')
print(f'ALL RUNS COMPLETE | Total time: {elapsed}')
print(f'{"="*60}')

In [ ]:
# Cell 10: Save results to Drive
results_path = '/content/outputs/all_results.json'
os.makedirs('/content/outputs', exist_ok=True)
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)

# Copy to Drive
drive_output_dir = os.path.join(PROJECT_DIR, 'outputs', 'colab_results')
if os.path.exists(drive_output_dir):
    shutil.rmtree(drive_output_dir)
shutil.copytree('/content/outputs', drive_output_dir)
print(f'Results saved to Drive: {drive_output_dir}')

In [ ]:
# Cell 11: Summary table
print(f'{"="*100}')
print(f'{"BCEDF Summary Table (cf. Thesis Table 6.1)":^100}')
print(f'{"="*100}')
print(f'{"Model":<15} {"Dataset":<15} {"Acc":<8} {"F1":<8} {"AUC":<8} {"Sens":<8} {"Spec":<8}')
print(f'{"-"*70}')
for run_name, r in results.items():
    print(f'{r["model"]:<15} {r["datasets"]:<15} {r["accuracy"]:<8.4f} {r["f1_score"]:<8.4f} {r["auc_roc"]:<8.4f} {r["sensitivity"]:<8.4f} {r["specificity"]:<8.4f}')
print(f'{"="*100}')